# Ingeniería del Dato: BASE 2 - María Duque Laredo

## Carga de librerías

In [62]:
import pandas as pd
from pathlib import Path

import pandas as pd 
from functools import reduce

import seaborn as sns

import matplotlib.pyplot as plt

import seaborn as sns

## Dataset 1: Gasto en educación por CCAA

In [38]:
# Carga del dataset 1: Gasto en educación por CCAA (INE)

gasto_educacion = pd.read_csv("../raw/gasto_hogares_educacion.csv", 
                         sep="\t", 
                         encoding="latin-1")

# Renombramos las columnas
gasto_educacion = gasto_educacion.rename(columns={
    "Comunidades y ciudades autónomas": "ccaa",
    "Tipo de gasto": "tipo_gasto",
    "Indicador": "indicador",
    "Total": "gasto_reglado_medio"
})

# De la variable 'ccaa' eliminamos la fila de 'Total nacional'

gasto_educacion = gasto_educacion[
    ~gasto_educacion['ccaa'].str.contains("Total", case=False)
]

# Eliminamos la variable 'indicador' porque no nos aporta nada

gasto_educacion = gasto_educacion.drop(columns=["indicador"])

# Normalizamos los nombres de las CCAA

reemplazos = {
    "Asturias, Principado de": "Asturias",
    "Balears, Illes": "Islas Baleares",
    "Castilla - La Mancha": "Castilla La Mancha",
    "Madrid, Comunidad de": "Madrid",
    "Murcia, Región de": "Murcia",
    "Navarra, Comunidad Foral de": "Navarra",
    "Rioja, La": "La Rioja",
    "Comunitat Valenciana": "Comunidad Valenciana",
    "Andalucía": "Andalucia",
    "Aragón": "Aragon",
    "Castilla y León": "Castilla y Leon",
    "País Vasco": "Pais Vasco"

}

gasto_educacion["ccaa"] = gasto_educacion["ccaa"].replace(reemplazos)

# Eliminamos la variable 'tipo_gasto' porque su valor es siempre el mismo 'Gasto en Servicios Educativos reglados'

gasto_educacion = gasto_educacion.drop(columns=['tipo_gasto'])

# Como la variable gasto_reglado_medio esta tipo string y no numérico, lo convertimos                                   

gasto_educacion['gasto_reglado_medio'] = pd.to_numeric(
    gasto_educacion['gasto_reglado_medio']
    .astype(str)
    .str.replace('.', '', regex=False),
    errors='coerce'
)

# Ordenamos el dataset para que se parezca al resto y sea más fácil el merge

gasto_educacion = gasto_educacion.sort_values(by='ccaa').reset_index(drop=True)

# Verificamos que no existen duplicados

print(
    "Duplicados:",
    gasto_educacion.duplicated(
        ["ccaa", "gasto_reglado_medio"]
    ).sum()
)

# Visualizamos el dataset limpio

display(gasto_educacion)

# Guardamos el dataset limpio

gasto_educacion.to_csv(
    "../processed/gasto_hogares_educacion_clean.csv",
    index=False
)

print(gasto_educacion['ccaa'].unique())
print(gasto_educacion.head())
print(gasto_educacion.columns)
print(gasto_educacion.shape)

Duplicados: 0


,ccaa,gasto_reglado_medio
0,Andalucia,1128
1,Aragon,9700
2,Asturias,8950
3,Canarias,1446
4,Cantabria,6800
5,Castilla La Mancha,8610
6,Castilla y Leon,1121
7,Cataluña,1945
8,Ceuta,3980
9,Comunidad Valenciana,1437


<StringArray>
[           'Andalucia',               'Aragon',             'Asturias',
             'Canarias',            'Cantabria',   'Castilla La Mancha',
      'Castilla y Leon',             'Cataluña',                'Ceuta',
 'Comunidad Valenciana',          'Extremadura',              'Galicia',
       'Islas Baleares',             'La Rioja',               'Madrid',
              'Melilla',               'Murcia',              'Navarra',
           'Pais Vasco']
Length: 19, dtype: str
        ccaa  gasto_reglado_medio
0  Andalucia                 1128
1     Aragon                 9700
2   Asturias                 8950
3   Canarias                 1446
4  Cantabria                 6800
Index(['ccaa', 'gasto_reglado_medio'], dtype='str')
(19, 2)


## Dataset 2: Centros educativos por CCAA

In [ ]:
# Carga del dataset 2: Centros educativos por CCAA (EducaBase)

n_centros_ccaa = pd.read_csv("../raw/centros_ccaa.csv", 
                         sep="\t", 
                         encoding="latin-1")

# Renombramos columnas

n_centros_ccaa = n_centros_ccaa.rename(columns={
    "Comunidad autónoma/provincia": "ccaa",
    "Tipo de centro": "tipo_centro",
    "Total": "n_centros"})

# Eliminamos columna 'Titularidad/financiación del centro', no es relevante para este dataset

n_centros_ccaa = n_centros_ccaa.drop(
    columns=["Titularidad/financiación del centro"]
)

# Limpiamos las Comunidades Autónomas
# 1. Eliminamos el código numérico delante de la CCAA

n_centros_ccaa["ccaa"] = n_centros_ccaa["ccaa"].str.replace(
    r"^\d+\s", "", regex=True
)

# 2. Normalizamos los nombres de las CCAA

n_centros_ccaa["ccaa"] = n_centros_ccaa["ccaa"].str.strip()

reemplazos_ccaa = {
    "ANDALUCÍA": "Andalucia",
    "ARAGÓN": "Aragon",
    "ASTURIAS, PRINCIPADO DE": "Asturias",
    "BALEARS, ILLES": "Islas Baleares",
    "CANARIAS": "Canarias",
    "CANTABRIA": "Cantabria",
    "CASTILLA Y LEÓN": "Castilla y Leon",
    "CASTILLA-LA MANCHA": "Castilla La Mancha",
    "CATALUÑA": "Cataluña",
    "COMUNITAT VALENCIANA": "Comunidad Valenciana",
    "EXTREMADURA": "Extremadura",
    "GALICIA": "Galicia",
    "MADRID, COMUNIDAD DE": "Madrid",
    "MURCIA, REGIÓN DE": "Murcia",
    "NAVARRA (Comunidad Foral de) (5)": "Navarra",
    "PAÍS VASCO": "Pais Vasco",
    "RIOJA, LA": "La Rioja",
    "CEUTA": "Ceuta",
    "MELILLA": "Melilla"
}

n_centros_ccaa["ccaa"] = n_centros_ccaa["ccaa"].replace(reemplazos_ccaa)

# Convertimos la variable 'n_centros' a numérica

n_centros_ccaa['n_centros'] = pd.to_numeric(
    n_centros_ccaa['n_centros']
    .astype(str)
    .str.replace('.', '', regex=False),
    errors='coerce'
)

# Eliminamos las filas de la variable 'tipo de centro' cuyo valor sea 'TOTAL'

n_centros_ccaa = n_centros_ccaa[
    n_centros_ccaa["tipo_centro"] != "TOTAL"
]

# Normalizamos los nombres de los tipos de centro

reemplazos_tipo = {
    "Centros E. Primaria (2)": "Centro_primaria",
    "Centros ESO y/o Bachillerato y/o FP (3)": "Centro_secundaria"
}

n_centros_ccaa["tipo_centro"] = n_centros_ccaa["tipo_centro"].replace(reemplazos_tipo)

# Transformamos los datos para pasarlos de formato 'long' (cada ccaa aparece en varias filas) a 'wide'(ccaa solo aparece una vez) pivotando el dataset

df_centros_pivot = n_centros_ccaa.pivot(
    index="ccaa",
    columns="tipo_centro",
    values="n_centros"
).reset_index()

# Eliminamos el nombre del indice 'tipo_centro' de las columnas 

df_centros_pivot.columns.name = None

# Vamos a calcular la proporción de cada tipo de centro por ccaa
# 1. Calculamos el número total de centros considerados por comunidad autónoma

df_centros_pivot["total_centros"] = (
    df_centros_pivot["Centro_primaria"] + df_centros_pivot["Centro_secundaria"]
)

# 2. Creamos variables relativas para evitar que el modelo sesgue por tamaño
# Primaria

df_centros_pivot["pct_centro_primaria"] = (
    df_centros_pivot["Centro_primaria"] / df_centros_pivot["total_centros"]
)

# Secundaria

df_centros_pivot["pct_centro_secundaria"] = (
    df_centros_pivot["Centro_secundaria"] / df_centros_pivot["total_centros"]
)

# Nos quedamos con las variables finales más útiles para el modelo

df_centros_final = df_centros_pivot[
    ["ccaa", "pct_centro_primaria", "pct_centro_secundaria"]
].copy()

# Verificamos que no haya nulos

print(df_centros_final.isnull().sum())

# Verificamos que no existen duplicados

print("Duplicados:", df_centros_final.duplicated(subset=["ccaa"]).sum())

# Visualizamos el dataset final limpio

display(df_centros_final)
print(df_centros_final.info())

# Guardamos el dataset limpio y pivotado

df_centros_final.to_csv(
    "../processed/centros_ccaa_clean.csv",
    index=False
)

ccaa                     0
pct_centro_primaria      0
pct_centro_secundaria    0
dtype: int64
Duplicados: 0


,ccaa,pct_centro_primaria,pct_centro_secundaria
0,Andalucia,0.142857,0.857143
1,Aragon,0.662844,0.337156
2,Asturias,0.668712,0.331288
3,Canarias,0.706564,0.293436
4,Cantabria,0.669767,0.330233
5,Castilla La Mancha,0.710583,0.289417
6,Castilla y Leon,0.680000,0.320000
7,Cataluña,0.182296,0.817704
8,Ceuta,0.708333,0.291667
9,Comunidad Valenciana,0.174560,0.825440


<class 'pandas.DataFrame'>
RangeIndex: 19 entries, 0 to 18
Data columns (total 3 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   ccaa                   19 non-null     str    
 1   pct_centro_primaria    19 non-null     float64
 2   pct_centro_secundaria  19 non-null     float64
dtypes: float64(2), str(1)
memory usage: 588.0 bytes
None


## Dataset 3: Graduados de la ESO por tipo de centro y CCAA

In [93]:
# Carga del dataset 3: Graduados de la ESO por tipo de centro y CCAA (EducaBase)

graduados_eso = pd.read_csv("../raw/graduados_eso_centro.csv", 
                         sep="\t", 
                         encoding="latin-1")

# Renombramos columnas

graduados_eso = graduados_eso.rename(columns={
    "Comunidad autónoma/provincia": "ccaa",
    "Tipo de promoción": "tipo_promocion",
    "Sexo": "sexo",
    "Curso": "curso",
    "Titularidad/financiación del centro": "tipo_centro",
    "Total": "pct_promociona_eso"
})

# Eliminamos columna 'tipo_promocion', no es relevante para este dataset

graduados_eso = graduados_eso.drop(
    columns=["tipo_promocion"]
)

# Eliminamos columna 'sexo', no es relevante para este dataset

graduados_eso = graduados_eso.drop(
    columns=["sexo"]
)

# Eliminamos columna 'curso', no es relevante para este dataset

graduados_eso = graduados_eso.drop(
    columns=["curso"]
)

# Limpiamos las Comunidades Autónomas
# 1. Eliminamos el código numérico delante de la CCAA

graduados_eso["ccaa"] = graduados_eso["ccaa"].str.replace(
    r"^\d+\s", "", regex=True
)

# 2. Normalizamos los nombres de las CCAA

graduados_eso["ccaa"] = graduados_eso["ccaa"].str.strip()

reemplazos_ccaa = {
    "ANDALUCÍA": "Andalucia",
    "ARAGÓN": "Aragon",
    "ASTURIAS, PRINCIPADO DE": "Asturias",
    "BALEARS, ILLES": "Islas Baleares",
    "CANARIAS": "Canarias",
    "CANTABRIA": "Cantabria",
    "CASTILLA Y LEÓN": "Castilla y Leon",
    "CASTILLA-LA MANCHA": "Castilla La Mancha",
    "CATALUÑA": "Cataluña",
    "COMUNITAT VALENCIANA": "Comunidad Valenciana",
    "EXTREMADURA": "Extremadura",
    "GALICIA": "Galicia",
    "MADRID, COMUNIDAD DE": "Madrid",
    "MURCIA, REGIÓN DE": "Murcia",
    "NAVARRA, COMUNIDAD FORAL DE": "Navarra",
    "PAÍS VASCO": "Pais Vasco",
    "RIOJA, LA": "La Rioja",
    "CEUTA": "Ceuta",
    "MELILLA": "Melilla"
}

graduados_eso["ccaa"] = graduados_eso["ccaa"].replace(reemplazos_ccaa)

# Convertimos y transformamos la variable 'pct_promociona_eso' a la adecuada

graduados_eso["pct_promociona_eso"] = (
    graduados_eso["pct_promociona_eso"]
    .astype(str)
    .str.replace(",", ".", regex=False)
)

graduados_eso["pct_promociona_eso"] = pd.to_numeric(graduados_eso["pct_promociona_eso"], errors="coerce")

# Normalizamos los valores de la variable 'tipo_centro'

graduados_eso["tipo_centro"] = graduados_eso["tipo_centro"].replace({
    "Centros públicos": "centro_publico",
    "Centros privados": "centro_privado",
    "Centros privados - Ense. concertada": "centro_concertado"
})

# Transformamos los datos para pasarlos de formato 'long' (cada ccaa aparece en varias filas) a 'wide'(ccaa solo aparece una vez) pivotando el dataset

df_eso_pivot = graduados_eso.pivot(
    index="ccaa",
    columns="tipo_centro",
    values="pct_promociona_eso"
).reset_index()

# Renombramos las columnas después de pivotar

df_eso_pivot = df_eso_pivot.rename(columns={
    "centro_publico": "eso_publico",
    "centro_privado": "eso_privado",
    "centro_concertado": "eso_concertado"
})

# Ordenamos el dataset para que tenga el mismo orden que el resto

df_eso_pivot = df_eso_pivot.sort_values("ccaa").reset_index(drop=True)

# Verificamos que no haya nulos

print(df_eso_pivot.isnull().sum())

# Verificamos que no existen duplicados

print("Duplicados:", df_eso_pivot.duplicated(subset=["ccaa"]).sum())

# Visualizamos el dataset final limpio

display(df_eso_pivot)
print(df_eso_pivot.info())

# Guardamos el dataset limpio y pivotado

df_eso_pivot.to_csv(
    "../processed/graduados_eso_centro_clean.csv",
    index=False
)

tipo_centro
ccaa              0
eso_concertado    0
eso_privado       0
eso_publico       0
dtype: int64
Duplicados: 0


tipo_centro,ccaa,eso_concertado,eso_privado,eso_publico
0,Andalucia,92.5,93.2,85.9
1,Aragon,90.5,91.1,87.1
2,Asturias,91.6,92.1,86.6
3,Canarias,95.3,95.9,85.0
4,Cantabria,94.4,94.4,91.3
5,Castilla La Mancha,90.4,91.2,83.9
6,Castilla y Leon,92.6,92.6,85.2
7,Cataluña,93.6,93.8,85.6
8,Ceuta,93.2,93.2,79.0
9,Comunidad Valenciana,92.3,92.7,82.7


<class 'pandas.DataFrame'>
RangeIndex: 19 entries, 0 to 18
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   ccaa            19 non-null     str    
 1   eso_concertado  19 non-null     float64
 2   eso_privado     19 non-null     float64
 3   eso_publico     19 non-null     float64
dtypes: float64(3), str(1)
memory usage: 740.0 bytes
None


## Dataset 4: Graduados de Bachillerato por tipo de centro y CCAA

In [15]:
# Carga del dataset 4: Graduados de Bachillerato por tipo de centro y CCAA (EducaBase)

graduados_bach = pd.read_csv("../raw/graduados_bach_centro.csv", 
                         sep="\t", 
                         encoding="latin-1")

# Renombramos las columnas

graduados_bach = graduados_bach.rename(columns={
    "Comunidad autónoma/provincia": "ccaa",
    "Sexo": "sexo",
    "Titularidad del centro": "tipo_centro",
    "Modalidad": "modalidad",
    "Total": "pct_promociona_bach"
})

# Eliminamos columna 'sexo', no es relevante para este dataset

graduados_bach = graduados_bach.drop(
    columns=["sexo"]
)

# Eliminamos columna 'modalidad', no es relevante para este dataset

graduados_bach = graduados_bach.drop(
    columns=["modalidad"]
)

# Limpiamos las Comunidades Autónomas
# 1. Eliminamos el código numérico delante de la CCAA

graduados_bach["ccaa"] = graduados_bach["ccaa"].str.replace(
    r"^\d+\s", "", regex=True
)

# 2. Normalizamos los nombres de las CCAA

graduados_bach["ccaa"] = graduados_bach["ccaa"].str.strip()

reemplazos_ccaa = {
    "ANDALUCÍA": "Andalucia",
    "ARAGÓN": "Aragon",
    "ASTURIAS, PRINCIPADO DE": "Asturias",
    "BALEARS, ILLES": "Islas Baleares",
    "CANARIAS": "Canarias",
    "CANTABRIA": "Cantabria",
    "CASTILLA Y LEÓN": "Castilla y Leon",
    "CASTILLA-LA MANCHA": "Castilla La Mancha",
    "CATALUÑA": "Cataluña",
    "COMUNITAT VALENCIANA": "Comunidad Valenciana",
    "EXTREMADURA": "Extremadura",
    "GALICIA": "Galicia",
    "MADRID, COMUNIDAD DE": "Madrid",
    "MURCIA, REGIÓN DE": "Murcia",
    "NAVARRA, COMUNIDAD FORAL DE": "Navarra",
    "PAÍS VASCO": "Pais Vasco",
    "RIOJA, LA (2)": "La Rioja",
    "CEUTA": "Ceuta",
    "MELILLA": "Melilla"
}

graduados_bach["ccaa"] = graduados_bach["ccaa"].replace(reemplazos_ccaa)

# Convertimos y transformamos la variable 'pct_promociona_bach' a la adecuada

graduados_bach["pct_promociona_bach"] = (
    graduados_bach["pct_promociona_bach"]
    .astype(str)
    .str.replace(",", ".", regex=False)
)

graduados_bach["pct_promociona_bach"] = pd.to_numeric(graduados_bach["pct_promociona_bach"], errors="coerce")

# Normalizamos la variable 'tipo_centro'

graduados_bach["tipo_centro"] = graduados_bach["tipo_centro"].replace({
    "CENTROS PÚBLICOS": "centro_publico",
    "CENTROS PRIVADOS": "centro_privado"
})

# Transformamos los datos para pasarlos de formato 'long' (cada ccaa aparece en varias filas) a 'wide'(ccaa solo aparece una vez) pivotando el dataset

df_bach_pivot = graduados_bach.pivot(
    index="ccaa",
    columns="tipo_centro",
    values="pct_promociona_bach"
).reset_index()

# Renombramos las columnas después de pivotar

df_bach_pivot = df_bach_pivot.rename(columns={
    "centro_publico": "pct_bach_publico",
    "centro_privado": "pct_bach_privado"
})

# Ordenamos el dataset

df_bach_pivot = df_bach_pivot.sort_values("ccaa").reset_index(drop=True)

# Verificamos que no haya nulos

print(df_bach_pivot.isnull().sum())

# Verificamos que no existen duplicados

print("Duplicados:", df_bach_pivot.duplicated(subset=["ccaa"]).sum())

# Visualizamos el dataset final limpio

display(df_bach_pivot)
print(df_bach_pivot.info())

# Guardamos el dataset limpio y pivotado

df_bach_pivot.to_csv(
    "../processed/graduados_bach_centro_clean.csv",
    index=False
)

print(df_bach_pivot.head())

tipo_centro
ccaa                0
pct_bach_privado    0
pct_bach_publico    0
dtype: int64
Duplicados: 0


tipo_centro,ccaa,pct_bach_privado,pct_bach_publico
0,Andalucia,95.3,87.4
1,Aragon,96.9,89.1
2,Asturias,95.5,85.2
3,Canarias,96.3,85.6
4,Cantabria,94.7,89.9
5,Castilla La Mancha,95.6,87.1
6,Castilla y Leon,95.4,85.1
7,Cataluña,96.2,89.4
8,Ceuta,95.7,74.7
9,Comunidad Valenciana,94.6,86.1


<class 'pandas.DataFrame'>
RangeIndex: 19 entries, 0 to 18
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ccaa              19 non-null     str    
 1   pct_bach_privado  19 non-null     float64
 2   pct_bach_publico  19 non-null     float64
dtypes: float64(2), str(1)
memory usage: 588.0 bytes
None
tipo_centro       ccaa  pct_bach_privado  pct_bach_publico
0            Andalucia              95.3              87.4
1               Aragon              96.9              89.1
2             Asturias              95.5              85.2
3             Canarias              96.3              85.6
4            Cantabria              94.7              89.9


## Dataset 5: Graduados FP Básica por centro y CCAA

In [27]:
# Carga del dataset 5: Graduados de FP Básica por tipo de centro y CCAA (EducaBase)

graduados_FPBasica = pd.read_csv("../raw/graduados_FPBasica_centro.csv", 
                         sep="\t", 
                         encoding="latin-1")

# Renombramos las columnas

graduados_FPBasica = graduados_FPBasica.rename(columns={
    "Comunidad autónoma/provincia": "ccaa",
    "Sexo": "sexo",
    "Titularidad del centro": "tipo_centro",
    "Total": "alumnos_fp_basica"
})

# Eliminamos columna 'sexo', no es relevante para este dataset

graduados_FPBasica = graduados_FPBasica.drop(
    columns=["sexo"]
)

# Limpiamos las Comunidades Autónomas
# 1. Eliminamos el código numérico delante de la CCAA

graduados_FPBasica["ccaa"] = graduados_FPBasica["ccaa"].str.replace(
    r"^\d+\s", "", regex=True
)

# 2. Normalizamos los nombres de las CCAA

graduados_FPBasica["ccaa"] = graduados_FPBasica["ccaa"].str.strip()

reemplazos_ccaa = {
    "ANDALUCÍA": "Andalucia",
    "ARAGÓN": "Aragon",
    "ASTURIAS, PRINCIPADO DE": "Asturias",
    "BALEARS, ILLES": "Islas Baleares",
    "CANARIAS": "Canarias",
    "CANTABRIA": "Cantabria",
    "CASTILLA Y LEÓN": "Castilla y Leon",
    "CASTILLA-LA MANCHA": "Castilla La Mancha",
    "CATALUÑA": "Cataluña",
    "COMUNITAT VALENCIANA": "Comunidad Valenciana",
    "EXTREMADURA": "Extremadura",
    "GALICIA": "Galicia",
    "MADRID, COMUNIDAD DE": "Madrid",
    "MURCIA, REGIÓN DE": "Murcia",
    "NAVARRA, COMUNIDAD FORAL DE": "Navarra",
    "PAÍS VASCO": "Pais Vasco",
    "RIOJA, LA": "La Rioja",
    "CEUTA": "Ceuta",
    "MELILLA": "Melilla"
}

graduados_FPBasica["ccaa"] = graduados_FPBasica["ccaa"].replace(reemplazos_ccaa)

# Limpiamos la variable 'alumnos_fp_basica'

graduados_FPBasica["alumnos_fp_basica"] = (
    graduados_FPBasica["alumnos_fp_basica"]
    .astype(str)
    .str.replace(".", "", regex=False)
    .astype(float)
)

# Normalizamos la variable 'tipo_centro'

graduados_FPBasica["tipo_centro"] = graduados_FPBasica["tipo_centro"].replace({
    "CENTROS PÚBLICOS": "centro_publico",
    "CENTROS PRIVADOS": "centro_privado",
    "TODOS LOS CENTROS": "total"
})

# Transformamos los datos para pasarlos de formato 'long' (cada ccaa aparece en varias filas) a 'wide'(ccaa solo aparece una vez) pivotando el dataset

df_fpbasica_pivot = graduados_FPBasica.pivot(
    index="ccaa",
    columns="tipo_centro",
    values="alumnos_fp_basica"
).reset_index()

# Dado que en este dataset no nos dan los % de alumnos graduados sino el número vamos a calcular el porcentaje

df_fpbasica_pivot["pct_fpbasica_publico"] = (
    df_fpbasica_pivot["centro_publico"] / df_fpbasica_pivot["total"]
)

df_fpbasica_pivot["pct_fpbasica_privado"] = (
    df_fpbasica_pivot["centro_privado"] / df_fpbasica_pivot["total"]
)

# Limpiamos el dataset, nos quedamos con las variables que ya son útiles

df_fpbasica_pivot = df_fpbasica_pivot[[
    "ccaa",
    "pct_fpbasica_publico",
    "pct_fpbasica_privado"
]]

# Ordenamos el dataset 

df_fpbasica_pivot = df_fpbasica_pivot.sort_values("ccaa").reset_index(drop=True)

# Verificamos que no haya nulos

print(df_fpbasica_pivot.isnull().sum())

# Verificamos que no existen duplicados

print("Duplicados:", df_fpbasica_pivot.duplicated(subset=["ccaa"]).sum())

# Visualizamos el dataset final limpio

display(df_fpbasica_pivot)
print(df_fpbasica_pivot.info())

# Guardamos el dataset limpio y pivotado

df_fpbasica_pivot.to_csv(
    "../processed/graduados_FPBasica_centro_clean.csv",
    index=False
)

tipo_centro
ccaa                    0
pct_fpbasica_publico    0
pct_fpbasica_privado    0
dtype: int64
Duplicados: 0


tipo_centro,ccaa,pct_fpbasica_publico,pct_fpbasica_privado
0,Andalucia,0.748701,0.251299
1,Aragon,0.691388,0.308612
2,Asturias,0.675676,0.324324
3,Canarias,0.938133,0.061867
4,Cantabria,0.592920,0.407080
5,Castilla La Mancha,0.088213,1.178651
6,Castilla y Leon,6.472125,3.527875
7,Cataluña,1.000000,0.000000
8,Ceuta,0.916667,0.083333
9,Comunidad Valenciana,0.884385,1.156152


<class 'pandas.DataFrame'>
RangeIndex: 19 entries, 0 to 18
Data columns (total 3 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   ccaa                  19 non-null     str    
 1   pct_fpbasica_publico  19 non-null     float64
 2   pct_fpbasica_privado  19 non-null     float64
dtypes: float64(2), str(1)
memory usage: 588.0 bytes
None


## Dataset 6: Graduados FP Medio por centro y CCAA

In [46]:
# Carga del dataset 6: Graduados de FP Medio por tipo de centro y CCAA (EducaBase)

graduados_FPMedio = pd.read_csv("../raw/graduados_FPMedio_centro.csv", 
                         sep="\t", 
                         encoding="latin-1")

# Renombramos las columnas

graduados_FPMedio = graduados_FPMedio.rename(columns={
    "Comunidad autónoma/provincia": "ccaa",
    "Sexo": "sexo",
    "Titularidad del centro": "tipo_centro",
    "Total": "alumnos_fp_medio"
})

# Eliminamos columna 'sexo', no es relevante para este dataset

graduados_FPMedio = graduados_FPMedio.drop(
    columns=["sexo"]
)

# Limpiamos las Comunidades Autónomas
# 1. Eliminamos el código numérico delante de la CCAA

graduados_FPMedio["ccaa"] = graduados_FPMedio["ccaa"].str.replace(
    r"^\d+\s", "", regex=True
)

# 2. Normalizamos los nombres de las CCAA

graduados_FPMedio["ccaa"] = graduados_FPMedio["ccaa"].str.strip()

reemplazos_ccaa = {
    "ANDALUCÍA": "Andalucia",
    "ARAGÓN": "Aragon",
    "ASTURIAS, PRINCIPADO DE": "Asturias",
    "BALEARS, ILLES": "Islas Baleares",
    "CANARIAS": "Canarias",
    "CANTABRIA": "Cantabria",
    "CASTILLA Y LEÓN": "Castilla y Leon",
    "CASTILLA-LA MANCHA": "Castilla La Mancha",
    "CATALUÑA": "Cataluña",
    "COMUNITAT VALENCIANA": "Comunidad Valenciana",
    "EXTREMADURA": "Extremadura",
    "GALICIA": "Galicia",
    "MADRID, COMUNIDAD DE": "Madrid",
    "MURCIA, REGIÓN DE": "Murcia",
    "NAVARRA, COMUNIDAD FORAL DE": "Navarra",
    "PAÍS VASCO": "Pais Vasco",
    "RIOJA, LA": "La Rioja",
    "CEUTA": "Ceuta",
    "MELILLA": "Melilla"
}

graduados_FPMedio["ccaa"] = graduados_FPMedio["ccaa"].replace(reemplazos_ccaa)

# Limpiamos la variable 'alumnos_fp_medio'

graduados_FPMedio["alumnos_fp_medio"] = (
    graduados_FPMedio["alumnos_fp_medio"]
    .astype(str)
    .str.replace(".", "", regex=False)
    .astype(float)
)

# Normalizamos la variable 'tipo_centro'

graduados_FPMedio["tipo_centro"] = graduados_FPMedio["tipo_centro"].replace({
    "CENTROS PÚBLICOS": "centro_publico",
    "CENTROS PRIVADOS": "centro_privado",
    "TODOS LOS CENTROS": "total"
})

# Transformamos los datos para pasarlos de formato 'long' (cada ccaa aparece en varias filas) a 'wide'(ccaa solo aparece una vez) pivotando el dataset

df_fpmedio_pivot = graduados_FPMedio.pivot(
    index="ccaa",
    columns="tipo_centro",
    values="alumnos_fp_medio"
).reset_index()

# Dado que en este dataset no nos dan los % de alumnos graduados sino el número vamos a calcular el porcentaje

df_fpmedio_pivot["pct_fpmedio_publico"] = (
    df_fpmedio_pivot["centro_publico"] / df_fpmedio_pivot["total"]
)

df_fpmedio_pivot["pct_fpmedio_privado"] = (
    df_fpmedio_pivot["centro_privado"] / df_fpmedio_pivot["total"]
)

# Limpiamos el dataset, nos quedamos con las variables que ya son útiles

df_fpmedio_pivot = df_fpmedio_pivot[[
    "ccaa",
    "pct_fpmedio_publico",
    "pct_fpmedio_privado"
]]

# Ordenamos el dataset 

df_fpmedio_pivot = df_fpmedio_pivot.sort_values("ccaa").reset_index(drop=True)

# Verificamos que no haya nulos

print(df_fpmedio_pivot.isnull().sum())

# Verificamos que no existen duplicados

print("Duplicados:", df_fpmedio_pivot.duplicated(subset=["ccaa"]).sum())

# Visualizamos el dataset final limpio

display(df_fpmedio_pivot)
print(df_fpmedio_pivot.info())

# Guardamos el dataset limpio y pivotado

df_fpmedio_pivot.to_csv(
    "../processed/graduados_FPMedio_centro_clean.csv",
    index=False
)

tipo_centro
ccaa                   0
pct_fpmedio_publico    0
pct_fpmedio_privado    0
dtype: int64
Duplicados: 0


tipo_centro,ccaa,pct_fpmedio_publico,pct_fpmedio_privado
0,Andalucia,0.611386,0.388614
1,Aragon,0.640554,0.359446
2,Asturias,0.670467,3.295331
3,Canarias,0.911922,0.880776
4,Cantabria,0.668857,3.311432
5,Castilla La Mancha,0.766401,0.233599
6,Castilla y Leon,6.622642,3.377358
7,Cataluña,0.691368,0.308632
8,Ceuta,1.000000,0.000000
9,Comunidad Valenciana,0.666828,0.033317


<class 'pandas.DataFrame'>
RangeIndex: 19 entries, 0 to 18
Data columns (total 3 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   ccaa                 19 non-null     str    
 1   pct_fpmedio_publico  19 non-null     float64
 2   pct_fpmedio_privado  19 non-null     float64
dtypes: float64(2), str(1)
memory usage: 588.0 bytes
None


## Dataset 7: Graduados FP Superior por centro y CCAA

In [73]:
# Carga del dataset 7: Graduados de FP Superior por tipo de centro y CCAA (EducaBase)

graduados_FPSuperior = pd.read_csv(
    "../raw/graduados_FPSuperior_centro.csv",
    sep="\t",
    encoding="latin-1"
)

# Renombramos las columnas

graduados_FPSuperior = graduados_FPSuperior.rename(columns={
    "Comunidad autónoma/provincia": "ccaa",
    "Sexo": "sexo",
    "Titularidad del centro": "tipo_centro",
    "Total": "alumnos_fp_superior"
})

# Eliminamos columna 'sexo', no es relevante para este dataset

graduados_FPSuperior = graduados_FPSuperior.drop(
    columns=["sexo"]
)

# Limpiamos las Comunidades Autónomas
# 1. Eliminamos el código numérico delante de la CCAA

graduados_FPSuperior["ccaa"] = graduados_FPSuperior["ccaa"].str.replace(
    r"^\d+\s", "", regex=True
)

# 2. Normalizamos los nombres de las CCAA

graduados_FPSuperior["ccaa"] = graduados_FPSuperior["ccaa"].str.strip()

reemplazos_ccaa = {
    "ANDALUCÍA": "Andalucia",
    "ARAGÓN": "Aragon",
    "ASTURIAS, PRINCIPADO DE": "Asturias",
    "BALEARS, ILLES": "Islas Baleares",
    "CANARIAS": "Canarias",
    "CANTABRIA": "Cantabria",
    "CASTILLA Y LEÓN": "Castilla y Leon",
    "CASTILLA-LA MANCHA": "Castilla La Mancha",
    "CATALUÑA": "Cataluña",
    "COMUNITAT VALENCIANA": "Comunidad Valenciana",
    "EXTREMADURA": "Extremadura",
    "GALICIA": "Galicia",
    "MADRID, COMUNIDAD DE": "Madrid",
    "MURCIA, REGIÓN DE": "Murcia",
    "NAVARRA, COMUNIDAD FORAL DE": "Navarra",
    "PAÍS VASCO": "Pais Vasco",
    "RIOJA, LA": "La Rioja",
    "CEUTA": "Ceuta",
    "MELILLA": "Melilla"
}

graduados_FPSuperior["ccaa"] = graduados_FPSuperior["ccaa"].replace(reemplazos_ccaa)

# Limpiamos la variable 'alumnos_fp_superior'

graduados_FPSuperior["alumnos_fp_superior"] = (
    graduados_FPSuperior["alumnos_fp_superior"]
    .astype(str)
    .str.replace(".", "", regex=False)
    .astype(float)
)

# Normalizamos la variable 'tipo_centro'

graduados_FPSuperior["tipo_centro"] = graduados_FPSuperior["tipo_centro"].replace({
    "CENTROS PÚBLICOS": "centro_publico",
    "CENTROS PRIVADOS": "centro_privado",
    "TODOS LOS CENTROS": "total"
})

# Transformamos los datos para pasarlos de formato 'long' (cada ccaa aparece en varias filas) a 'wide'(ccaa solo aparece una vez) pivotando el dataset

df_fpsuperior_pivot = graduados_FPSuperior.pivot(
    index="ccaa",
    columns="tipo_centro",
    values="alumnos_fp_superior"
).reset_index()

# Dado que en este dataset no nos dan los % de alumnos graduados sino el número vamos a calcular el porcentaje

df_fpsuperior_pivot["pct_fpsuperior_publico"] = (
    df_fpsuperior_pivot["centro_publico"] / df_fpsuperior_pivot["total"]
)

df_fpsuperior_pivot["pct_fpsuperior_privado"] = (
    df_fpsuperior_pivot["centro_privado"] / df_fpsuperior_pivot["total"]
)

# Limpiamos el dataset, nos quedamos con las variables que ya son útiles

df_fpsuperior_pivot = df_fpsuperior_pivot[[
    "ccaa",
    "pct_fpsuperior_publico",
    "pct_fpsuperior_privado"
]]

# Ordenamos el dataset 

df_fpsuperior_pivot = df_fpsuperior_pivot.sort_values("ccaa").reset_index(drop=True)

# Verificamos que no haya nulos

print(df_fpsuperior_pivot.isnull().sum())

# Verificamos que no existen duplicados

print("Duplicados:", df_fpsuperior_pivot.duplicated(subset=["ccaa"]).sum())

# Visualizamos el dataset final limpio

display(df_fpsuperior_pivot)
print(df_fpsuperior_pivot.info())

# Guardamos el dataset limpio y pivotado

df_fpsuperior_pivot.to_csv(
    "../processed/graduados_FPSuperior_centro_clean.csv",
    index=False
)

print(df_fpsuperior_pivot.head())

tipo_centro
ccaa                      0
pct_fpsuperior_publico    0
pct_fpsuperior_privado    0
dtype: int64
Duplicados: 0


tipo_centro,ccaa,pct_fpsuperior_publico,pct_fpsuperior_privado
0,Andalucia,0.593634,0.406366
1,Aragon,0.585604,0.414396
2,Asturias,0.745611,2.543894
3,Canarias,0.869347,1.306533
4,Cantabria,0.705781,2.942187
5,Castilla La Mancha,0.853841,1.461592
6,Castilla y Leon,0.716542,0.283458
7,Cataluña,0.062148,0.378524
8,Ceuta,1.000000,0.000000
9,Comunidad Valenciana,0.680974,0.031903


<class 'pandas.DataFrame'>
RangeIndex: 19 entries, 0 to 18
Data columns (total 3 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   ccaa                    19 non-null     str    
 1   pct_fpsuperior_publico  19 non-null     float64
 2   pct_fpsuperior_privado  19 non-null     float64
dtypes: float64(2), str(1)
memory usage: 588.0 bytes
None
tipo_centro       ccaa  pct_fpsuperior_publico  pct_fpsuperior_privado
0            Andalucia                0.593634                0.406366
1               Aragon                0.585604                0.414396
2             Asturias                0.745611                2.543894
3             Canarias                0.869347                1.306533
4            Cantabria                0.705781                2.942187


## Dataset 8: Renta per cápita por CCAA - Preparar para el merge

In [79]:
# Carga del dataset 8: Renta per cápita por CCAA ya la teníamos limpia, ahora la vamos a preparar para el merge

renta_ccaa = pd.read_csv(
    "../processed/renta_ccaa_clean.csv"
)

# Filtramos únicamente el año 2023 para mantener coherencia temporal con el resto de variables del modelo de clustering

renta_ccaa = renta_ccaa[renta_ccaa["year"] == 2023].copy()

# Limpiamos el dataset para quedarnos solo con las variables necesarias para el merge final

renta_ccaa = renta_ccaa[["ccaa", "renta_pc"]].copy()

# Renombramos la variable 'renta_pc' para que quede más clara

renta_ccaa = renta_ccaa.rename(columns={
    "renta_pc": "renta_per_capita"
})

# Verificamos que no haya nulos

print(renta_ccaa.isnull().sum())

# Verificamos que no existen duplicados

print("Duplicados:", renta_ccaa.duplicated(subset=["ccaa"]).sum())

# Visualizamos el dataset final limpio

display(renta_ccaa)
print(renta_ccaa.info())

# Guardamos el dataset limpio y pivotado

renta_ccaa.to_csv(
    "../processed/rentapercapita2_ccaa_clean.csv",
    index=False
)

print(renta_ccaa.head())

ccaa                0
renta_per_capita    0
dtype: int64
Duplicados: 0


,ccaa,renta_per_capita
4,Andalucia,11.719
11,Aragon,14.810
18,Asturias,15.432
25,Canarias,12.177
32,Cantabria,14.162
39,Castilla - La Mancha,11.913
46,Castilla y Leon,14.124
53,Cataluña,15.830
60,Ceuta,13.421
67,Comunidad Valenciana,12.805


<class 'pandas.DataFrame'>
RangeIndex: 19 entries, 4 to 130
Data columns (total 2 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ccaa              19 non-null     str    
 1   renta_per_capita  19 non-null     float64
dtypes: float64(1), str(1)
memory usage: 436.0 bytes
None
         ccaa  renta_per_capita
4   Andalucia            11.719
11     Aragon            14.810
18   Asturias            15.432
25   Canarias            12.177
32  Cantabria            14.162
